# COMP8420 Assignment 3: Advanced LLM & RAG Techniques
This notebook demonstrates the complete implementation of **all 6 Advanced NLP/LLM Techniques** in our E-commerce system.

### 1. Foundation Models & Fallback Orchestration
Connects and manages Gemma & Llama local model requests.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from src.ollama_client import OllamaOrchestrator
ollama = OllamaOrchestrator()
models = ollama.get_available_models()
print('Available Local Models:', models)

Available Ollama models: ['gemma2:2b', 'llama3.2:3b', 'llama3.2:latest', 'finance-llama:latest', 'finance-qwen:latest', 'finance-critic:latest', 'finance-advisor:latest', 'qwen2.5:14b', 'phi3:latest', 'llama3.1:latest']
Available Ollama models: ['gemma2:2b', 'llama3.2:3b', 'llama3.2:latest', 'finance-llama:latest', 'finance-qwen:latest', 'finance-critic:latest', 'finance-advisor:latest', 'qwen2.5:14b', 'phi3:latest', 'llama3.1:latest']
Available Local Models: ['gemma2:2b', 'llama3.2:3b', 'llama3.2:latest', 'finance-llama:latest', 'finance-qwen:latest', 'finance-critic:latest', 'finance-advisor:latest', 'qwen2.5:14b', 'phi3:latest', 'llama3.1:latest']


### 2. SQLite RAG Grounding datastore
Index product metadata and retrieve review contexts for prompts.

In [2]:
from src.RAG_engine import RAGEngine
db = RAGEngine(db_path='../data/ecommerce_rag.db')
context = db.retrieve_context_for_rag(product_id='AV1YnRtnglJLPUi8IJmV', limit=2)
print('Retrieved RAG Grounding context:\n', context)

Retrieved RAG Grounding context:
 Review #1 by BobBrews (Rating: 5/5 stars):
"Easy to use and fast on. Great for sun or night A+++"

Review #2 by dlkween (Rating: 5/5 stars):
"great value. it doesn't hurt my eyes. So easy to use and upload books. best of all my kids wont try and use it to surf the web."




### 3. Chain-of-Thought (CoT) grounded Copywriter
Generates professional descriptions using Gemma with structural CoT prompts.

In [3]:
from src.generator import GroundedProductDescGenerator
generator = GroundedProductDescGenerator(ollama)
product_meta = db.get_product('AV1YnRtnglJLPUi8IJmV')
desc = generator.generate_description(product_meta, context)
print('Generated Grounded Description:\n', desc)

Generated Grounded Description:
 **Kindle Voyage E-reader - Reimagined for E-Commerce Excellence**

This Kindle Voyage e-reader is more than just a book reader; it's an experience.  Its 6-inch high-resolution display with adaptive built-in light delivers vivid, crystal clear text that's comfortable to read even in bright sunlight or dim evening conditions. PagePress sensors ensure precise page turns, and the intuitive design makes navigation effortless, whether you're browsing for your next read or managing your digital library. 

The Kindle Voyage is a testament to seamless performance. Buyers consistently praise its speed, ease of use, and long battery life. Its sleek and compact design makes it perfect for carrying anywhere, from bedside tables to crowded commutes.  Users also rave about the built-in Wi-Fi connectivity which opens up a world of digital content at your fingertips. 

From enjoying novels on vacation to staying organized with reading lists, the Kindle Voyage delivers e

### 4. Few-Shot In-Context Classifier
Teaching Llama to detect AI-written reviews using historical examples.

In [4]:
from src.traditional_ml import TraditionalMLClassifier
from src.fraud_detection import AuthenticityFraudEngine
ml_model = TraditionalMLClassifier()
fraud_engine = AuthenticityFraudEngine(ml_model, ollama)
test_review = "This product is an absolute masterpiece of modern audio engineering. The sleek design is outstanding!"
ai_score = fraud_engine.classify_ai_review_llm(test_review)
print('AI-Written confidence score (0.0 to 1.0):', ai_score)

AI-Written confidence score (0.0 to 1.0): 0.2


### 5. Multi-dimensional Ensemble fraud index
Combines stylometrics, timestamp frequencies, rating-sentiment conflicts, and LLMs.

In [5]:
suspicious_review = {
    'text': 'Absolute garbage! Broke immediately on day one.',
    'rating': 5, # High ratings contradicts severe complaints
    'username': '', # Anonymous reviewer
    'timestamp': 1782298800
}
metrics = fraud_engine.evaluate_fraud_index(suspicious_review)
print('Unified Fraud Index Score:', metrics['fraud_score'], '%')
print('Detected Fraud indicators:', metrics['flags'])

Unified Fraud Index Score: 40 %
Detected Fraud indicators: {'low_lexical_diversity': False, 'excessive_shouting': False, 'sentiment_rating_contradiction': True, 'anonymous_reviewer': True, 'burst_posting_rate': False}


### 6. LLM-as-a-Judge Scorecard
Independent QA evaluation of copy accuracy and tone.

In [6]:
import json
from src.evaluator import LLMasJudgeEvaluator
judge = LLMasJudgeEvaluator(ollama)
scorecard = judge.evaluate_description(product_meta, desc)
print('LLM-as-a-judge Scorecard:\n', json.dumps(scorecard, indent=2))

Strict JSON loads failed (Expecting ',' delimiter: line 13 column 4 (char 757)). Running Regex Fallback Extractor...
LLM-as-a-judge Scorecard:
 {
  "factual_consistency": {
    "score": 4,
    "reasoning": "The generated description accurately conveys the features and benefits of the Kindle Voyage E-reader, but introduces some minor creative liberties with phrases like 'Reimagined for E-Commerce Excellence' that are not present in the original specs."
  },
  "hallucination_rate": {
    "score": 5,
    "reasoning": "The generated description does not contain any hallucinations or false claims about the product's features or specifications."
  },
  "tonal_professionalism": {
    "score": 4,
    "reasoning": "The tone of the generated description is professional and persuasive, effectively conveying the benefits and value proposition of the Kindle Voyage E-reader to potential customers."
  },
  "overall_audit_verdict": "APPROVED_FOR_STOREFRONT"
}
